In [ ]:
# 安装 LangChain 和 OpenAI 相关依赖
!pip install -U langchain langchain-openai
# 安装 Graphviz 图形可视化工具（系统级别）
!apt-get install -y graphviz graphviz-dev
# 安装 Python 的 Graphviz 绑定库
!pip install pygraphviz

In [ ]:
# 导入类型定义和图构建相关模块
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. 定义 State（状态数据结构）
class HelloState(TypedDict):
    name: str        # 用户姓名
    greeting: str    # 问候语

# 2. 定义节点（每个节点是一个处理函数）
def greet(state: HelloState) -> dict:
    """生成问候语节点"""
    name = state["name"]
    return {"greeting": f"你好, {name} !"}

def add_emoji(state: HelloState) -> dict:
    """在问候语后添加表情符号的节点"""
    greeting = state["greeting"]
    return {"greeting": greeting + " 👋"}

# 3. 构建图（定义节点和边的连接关系）
graph = StateGraph(HelloState)

# 添加节点到图中
graph.add_node("greet", greet)
graph.add_node("add_emoji", add_emoji)

# 定义节点之间的执行顺序（边）
graph.add_edge(START, "greet")        # 起始点 -> 问候节点
graph.add_edge("greet", "add_emoji")  # 问候节点 -> 添加表情节点
graph.add_edge("add_emoji", END)      # 添加表情节点 -> 结束

In [ ]:
# 4. 编译图（将图转换为可执行的应用）
app = graph.compile()

In [ ]:
# 5. 运行（传入初始状态并执行图）
result = app.invoke({"name": "张三"})
print(result["greeting"])  # 输出：你好, 张三！ 👋

# 导入图像显示工具
from IPython.display import Image, display
import os

# 保存图表为 PNG 文件（可视化执行流程）
try:
    # 获取图的可视化数据，xray=True 显示详细信息
    png_data = app.get_graph(xray=True).draw_png()
    output_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "graph_diagram.png")
    with open(output_path, "wb") as f:
        f.write(png_data)
    print(f"图表已保存到: {output_path}")

    # 同时在 notebook 中显示
    display(Image(png_data))
except Exception as e:
    # 如果 Graphviz 不可用，使用 Mermaid 文本格式替代显示
    print(f"Graphviz 渲染失败: {e}")
    print("\n使用 Mermaid 文本方式显示:")
    print(app.get_graph(xray=True).draw_mermaid())